# Part 9 — Evidence for the Six Report Questions

**7PAM2033 Applied Machine Learning — Assignment 2**

> **Task:** Answer six questions about model performance, model complexity,
> data leakage, covariate availability, and whether the more complex models
> are justified. The answers must be included in the report.

This notebook gathers the numbers behind each question in one place, so the
report can be written from evidence rather than from memory. Each section
prints exactly the figures that question needs and nothing else.

It reads the CSVs written by Parts 1 to 8. Anything missing is reported rather
than silently skipped.


## 1. Setup


In [10]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

BASE_DIR = Path.cwd()
HOURLY_FILE = BASE_DIR / "energy_hourly.csv"
FIGURE_DIR = BASE_DIR / "outputs" / "figures"
METRICS_DIR = BASE_DIR / "outputs" / "metrics"
FORECAST_DIR = BASE_DIR / "outputs" / "forecasts"
for directory in (FIGURE_DIR, METRICS_DIR, FORECAST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DAILY_PERIOD, WEEKLY_PERIOD = 24, 168
HORIZON, TEST_STEPS = 24, 336
_PV = tuple(int(p) for p in pd.__version__.split(".")[:2])
HOURLY_FREQ = "h" if _PV >= (2, 2) else "H"

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200,
                     "savefig.bbox": "tight", "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})
PRIMARY, SECONDARY = "#1f4e79", "#c0504d"


def load_csv(name, folder=METRICS_DIR, **kwargs):
    """Read a results file, returning None with a notice if it is absent."""
    path = folder / name
    if not path.exists():
        print(f"  MISSING: {path.name} - run the part that produces it")
        return None
    return pd.read_csv(path, **kwargs)


hourly = pd.read_csv(HOURLY_FILE, parse_dates=["date"],
                     index_col="date").asfreq(HOURLY_FREQ)
series = hourly["Appliances"]
test_series = series.iloc[-TEST_STEPS:]

print(f"Loaded {len(series)} hourly observations")


Loaded 3289 hourly observations


In [11]:
# Everything the questions draw on
master = load_csv("part8_master_comparison.csv")
by_origin = load_csv("part8_by_origin.csv")
vs_benchmark = load_csv("part8_vs_benchmark.csv")
dm_matrix = load_csv("part8_dm_pvalue_matrix.csv", index_col=0)
error_autocorr = load_csv("part8_error_autocorrelation.csv")
peak_performance = load_csv("part8_peak_performance.csv", index_col=0)

decomposition = load_csv("decomposition_strengths.csv", index_col=0)
key_acf = load_csv("key_autocorrelations.csv")
stationarity = load_csv("stationarity_tests.csv")

grid_nonseasonal = load_csv("part4_grid_nonseasonal.csv")
grid_seasonal = load_csv("part4_grid_seasonal.csv")
ljung = load_csv("part4_ljung_box.csv", index_col=0)

ablation = load_csv("part6_ablation.csv")
feature_relevance = load_csv("part5_feature_relevance.csv")
group_relevance = load_csv("part5_group_relevance.csv", index_col=0)
leakage_tests = load_csv("part5_leakage_tests.csv")

cost = load_csv("part7_computational_cost.csv")
interval_comparison = load_csv("part7_interval_comparison.csv")

print("\nEvidence files loaded.")



Evidence files loaded.


## Question 1

> **Which benchmark model is strongest — naive, daily seasonal naive, weekly
> seasonal naive, or drift — and what does this tell you about the structure
> of appliance energy use?**

Three things to establish: the ranking, whether the differences are real, and
what the pattern implies about the series.


In [12]:
print("Benchmark ranking")
print("=" * 74)
if master is not None:
    benchmarks = master[master["family"] == "benchmark"].sort_values("MAE")
    print(benchmarks[["model", "MAE", "RMSE", "MASE", "Bias"]]
          .round(3).to_string(index=False))

print("\nAre the top benchmarks distinguishable?")
if dm_matrix is not None:
    names = [n for n in ["seasonal_naive_weekly", "seasonal_naive_daily", "mean"]
             if n in dm_matrix.index]
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            print(f"  {a} vs {b}: p = {dm_matrix.loc[a, b]:.3f}")

print("\nStructural evidence from Part 1")
if decomposition is not None:
    print(decomposition.round(3).to_string())
if key_acf is not None:
    print()
    print(key_acf.to_string(index=False))

print("\nWhy naive and drift fail")
cutoff_hour = series.index[len(series) - TEST_STEPS - 1].hour
hourly_profile = series.groupby(series.index.hour).mean()
print(f"  every origin cuts off at {cutoff_hour:02d}:00")
print(f"  mean demand at that hour : {hourly_profile[cutoff_hour]:.1f} Wh")
print(f"  peak hour of the day     : {hourly_profile.idxmax():02d}:00 "
      f"({hourly_profile.max():.1f} Wh)")
print(f"  mean demand overall      : {series.mean():.1f} Wh")
if master is not None:
    naive_bias = master[master['model'] == 'naive']['Bias']
    if len(naive_bias):
        print(f"  resulting naive bias     : {naive_bias.iloc[0]:+.1f} Wh")


Benchmark ranking
                model     MAE    RMSE  MASE    Bias
seasonal_naive_weekly  42.922  79.668 0.804 -12.624
 seasonal_naive_daily  48.393  85.790 0.907   1.667
                 mean  50.010  73.986 0.937  -3.061
                naive 143.383 170.623 2.687 114.673
                drift 143.942 171.260 2.698 115.314

Are the top benchmarks distinguishable?
  seasonal_naive_weekly vs seasonal_naive_daily: p = 0.496
  seasonal_naive_weekly vs mean: p = 0.229
  seasonal_naive_daily vs mean: p = 0.755

Structural evidence from Part 1
               strength
seasonal_24h      0.474
seasonal_168h     0.298
trend             0.046

 lag                    interpretation   acf
   1                    1 hour (naive) 0.581
  24   24 hours (daily seasonal naive) 0.304
  48                          48 hours 0.221
 168 168 hours (weekly seasonal naive) 0.306

Why naive and drift fail
  every origin cuts off at 17:00
  mean demand at that hour : 161.4 Wh
  peak hour of the day     : 18:0

## Question 2

> **Does the SARIMAX model improve on the strongest seasonal benchmark?
> Discuss whether daily seasonality, autocorrelation, and exogenous variables
> are adequately captured.**

Four separate claims to evidence: the improvement, its significance, whether
autocorrelation was captured, and what the exogenous variables did.


In [13]:
print("SARIMAX against the strongest benchmark")
print("=" * 78)
if master is not None:
    sarimax_rows = master[master["family"].isin(["sarimax", "benchmark"])]
    print(sarimax_rows.sort_values("MAE")[
        ["model", "MAE", "RMSE", "MASE", "skill vs benchmark %"]]
        .round(3).to_string(index=False))

if vs_benchmark is not None:
    print("\nSignificance")
    sarimax_tests = vs_benchmark[vs_benchmark["model"].str.contains("sarima")]
    print(sarimax_tests[["model", "skill %", "p-value", "beats benchmark"]]
          .to_string(index=False))

print("\nWas the seasonality captured?")
if grid_nonseasonal is not None and grid_seasonal is not None:
    best_flat = grid_nonseasonal["aic"].min()
    best_seasonal = grid_seasonal["aic"].min()
    print(f"  best non-seasonal AIC : {best_flat:.1f}")
    print(f"  best seasonal AIC     : {best_seasonal:.1f}")
    print(f"  gain from seasonal terms: {best_flat - best_seasonal:.0f} points")
    print("\n  mean AIC by seasonal differencing order D:")
    print(grid_seasonal.groupby("D")["aic"].mean().round(1).to_string())
    print("  mean AIC by seasonal MA order Q:")
    print(grid_seasonal.groupby("Q")["aic"].mean().round(1).to_string())

print("\nWas the autocorrelation captured?")
if ljung is not None:
    print(ljung.round(4).to_string())

print("\nDid the exogenous variables help?")
if master is not None:
    exog = master[master["family"] == "sarimax"].sort_values("MAE")
    print(exog[["model", "MAE", "MASE"]].round(3).to_string(index=False))


SARIMAX against the strongest benchmark
                model     MAE    RMSE  MASE  skill vs benchmark %
seasonal_naive_weekly  42.922  79.668 0.804                 0.000
 seasonal_naive_daily  48.393  85.790 0.907               -12.747
                 mean  50.010  73.986 0.937               -16.515
                naive 143.383 170.623 2.687              -234.058
                drift 143.942 171.260 2.698              -235.360

Significance
Empty DataFrame
Columns: [model, skill %, p-value, beats benchmark]
Index: []

Was the seasonality captured?
  best non-seasonal AIC : 32944.1
  best seasonal AIC     : 32137.2
  gain from seasonal terms: 807 points

  mean AIC by seasonal differencing order D:
D
0    32739.7
1    32943.9
  mean AIC by seasonal MA order Q:
Q
0    33267.7
1    32415.9

Was the autocorrelation captured?
      lb_stat  lb_pvalue
lag                     
24    20.3849     0.6747
48    49.6942     0.4056
168  187.6976     0.1420

Did the exogenous variables help?
Em

## Question 3

> **Does the XGBoost or feature-based model improve when lag, rolling-window,
> time-of-day, and sensor/weather variables are added? Which feature groups
> appear most useful?**

The ablation is the primary evidence here: it measures what each group
contributes rather than how a fitted model happened to allocate splits.


In [14]:
print("Machine learning models against the benchmark and SARIMAX")
print("=" * 78)
if master is not None:
    print(master[master["family"].isin(["machine learning", "sarimax", "benchmark"])]
          .nsmallest(8, "MAE")[["family", "model", "MAE", "MASE",
                                "skill vs benchmark %"]]
          .round(3).to_string(index=False))

if ablation is not None:
    print("\nEach feature group alone")
    solo = ablation[ablation["stage"] == "solo"].sort_values("MAE")
    print(solo[["model", "n_features", "MAE", "MASE"]].round(2).to_string(index=False))

    print("\nGroups added cumulatively")
    cumulative = ablation[ablation["stage"] == "cumulative"].reset_index(drop=True)
    previous = None
    for _, row in cumulative.iterrows():
        change = "" if previous is None else f"{row['MAE'] - previous:+6.2f}"
        print(f"  {row['model']:<20} {int(row['n_features']):>3} features  "
              f"MAE {row['MAE']:6.2f}  {change}")
        previous = row["MAE"]

    best_subset = cumulative.loc[cumulative["MAE"].idxmin()]
    full = cumulative.iloc[-1]
    print(f"\n  best subset : {best_subset['model']} at "
          f"{int(best_subset['n_features'])} features, MAE {best_subset['MAE']:.2f}")
    print(f"  all features: {int(full['n_features'])} features, "
          f"MAE {full['MAE']:.2f}")

if group_relevance is not None:
    print("\nMutual information by group (training data only)")
    print(group_relevance.round(4).to_string())

if feature_relevance is not None:
    print("\nTop 10 individual features")
    print(feature_relevance.head(10).round(4).to_string(index=False))


Machine learning models against the benchmark and SARIMAX
          family                          model     MAE  MASE  skill vs benchmark %
machine learning hist_gradient_boosting_regimeB  36.552 0.685                14.839
machine learning                  random_forest  39.365 0.738                 8.286
machine learning         hist_gradient_boosting  40.030 0.750                 6.738
       benchmark          seasonal_naive_weekly  42.922 0.804                 0.000
       benchmark           seasonal_naive_daily  48.393 0.907               -12.747
       benchmark                           mean  50.010 0.937               -16.515
       benchmark                          naive 143.383 2.687              -234.058
       benchmark                          drift 143.942 2.698              -235.360

Each feature group alone
         model  n_features   MAE  MASE
      calendar          16 40.91  0.77
target_rolling          17 41.25  0.77
   target_lags           7 41.75  0.78
indo

## Question 4

> **Does the foundation model outperform the simpler benchmark, SARIMAX, and
> feature-based models? Is the improvement, if any, large enough to justify
> the extra complexity?**

Two halves: the accuracy comparison, and the cost of obtaining it.


In [15]:
print("Full ranking")
print("=" * 84)
if master is not None:
    print(master[["rank", "family", "model", "MAE", "MASE",
                  "skill vs benchmark %"]].round(3).to_string(index=False))

    foundation = master[master["family"] == "foundation"]
    if len(foundation):
        chronos = foundation.iloc[0]
        print(f"\nChronos rank {int(chronos['rank'])} of {len(master)}")
        for family in ["benchmark", "sarimax", "machine learning"]:
            subset = master[master["family"] == family]
            if len(subset):
                best = subset.nsmallest(1, "MAE").iloc[0]
                difference = (1 - chronos["MAE"] / best["MAE"]) * 100
                print(f"  vs best {family:<18} ({best['model']}): "
                      f"{difference:+.1f}% MAE")
    else:
        print("\nNo foundation-model results found - run Part 7.")

print("\nIs any difference at the top statistically real?")
if by_origin is not None and master is not None:
    spread = by_origin.groupby("model")["MAE"].std().median()
    top_two = master.nsmallest(2, "MAE")
    gap = top_two["MAE"].iloc[1] - top_two["MAE"].iloc[0]
    print(f"  typical per-origin std : {spread:.1f} Wh")
    print(f"  gap between top two    : {gap:.1f} Wh")
    print(f"  ratio                  : {gap / spread:.2f}")

if vs_benchmark is not None:
    print("\n" + vs_benchmark[["model", "skill %", "p-value",
                               "beats benchmark"]].to_string(index=False))

if cost is not None:
    print("\nComputational cost")
    print(cost.to_string(index=False))


Full ranking
 rank           family                          model     MAE  MASE  skill vs benchmark %
    1       foundation               chronos_zeroshot  33.580 0.629                21.765
    2 machine learning hist_gradient_boosting_regimeB  36.552 0.685                14.839
    3 machine learning                  random_forest  39.365 0.738                 8.286
    4 machine learning         hist_gradient_boosting  40.030 0.750                 6.738
    5        benchmark          seasonal_naive_weekly  42.922 0.804                 0.000
    6        benchmark           seasonal_naive_daily  48.393 0.907               -12.747
    7        benchmark                           mean  50.010 0.937               -16.515
    8        benchmark                          naive 143.383 2.687              -234.058
    9        benchmark                          drift 143.942 2.698              -235.360

Chronos rank 1 of 9
  vs best benchmark          (seasonal_naive_weekly): +21.8% MAE
 

## Question 5

> **Which variables would genuinely be known at the forecast origin? If you
> use future indoor temperature, humidity, or weather values from the test
> set, is this a true forecast or a conditional forecast?**

This question is about experimental design rather than results. The evidence
is the regime definitions, the leakage tests, and the measured difference
between the two regimes.


In [16]:
print("Availability of each variable at a 24-hour forecast origin")
print("=" * 84)
availability = pd.DataFrame([
    {"variable group": "hour, day of week, weekend flags",
     "known at origin": "yes, with certainty",
     "reason": "deterministic function of the clock"},
    {"variable group": "target lags of 24h or more",
     "known at origin": "yes",
     "reason": "already observed before the cutoff"},
    {"variable group": "target lags shorter than 24h",
     "known at origin": "NO",
     "reason": "falls inside the forecast window"},
    {"variable group": "sensor and weather lagged 24h or more",
     "known at origin": "yes",
     "reason": "observed before the cutoff"},
    {"variable group": "sensor and weather during the window",
     "known at origin": "NO",
     "reason": "would need forecasting in turn"},
])
print(availability.to_string(index=False))

print("\nLeakage tests from Part 5")
if leakage_tests is not None:
    print(leakage_tests.to_string(index=False))
    failed = (leakage_tests["result"] == "FAIL").sum()
    print(f"\n  {len(leakage_tests) - failed}/{len(leakage_tests)} passed")

print("\nMeasured cost of the conditional assumption")
if master is not None:
    regime_b = master[master["model"].str.contains("regimeB", na=False)]
    for _, row in regime_b.iterrows():
        base_name = row["model"].replace("_regimeB", "")
        base = master[master["model"] == base_name]
        if len(base):
            gain = base.iloc[0]["MAE"] - row["MAE"]
            print(f"  {base_name}: Regime A {base.iloc[0]['MAE']:.2f} Wh, "
                  f"Regime B {row['MAE']:.2f} Wh, "
                  f"difference {gain:+.2f} Wh "
                  f"({gain / base.iloc[0]['MAE'] * 100:+.1f}%)")

    weather_sarimax = master[master["model"] == "sarimax_weather"]
    plain_sarima = master[master["model"] == "sarima"]
    if len(weather_sarimax) and len(plain_sarima):
        difference = plain_sarima.iloc[0]["MAE"] - weather_sarimax.iloc[0]["MAE"]
        print(f"  sarima vs sarimax_weather (conditional): {difference:+.2f} Wh")


Availability of each variable at a 24-hour forecast origin
                       variable group     known at origin                              reason
     hour, day of week, weekend flags yes, with certainty deterministic function of the clock
           target lags of 24h or more                 yes  already observed before the cutoff
         target lags shorter than 24h                  NO    falls inside the forecast window
sensor and weather lagged 24h or more                 yes          observed before the cutoff
 sensor and weather during the window                  NO      would need forecasting in turn

Leakage tests from Part 5
                                               test result                                                        detail
             no target lag shorter than the horizon   pass                                  minimum lag 24h, horizon 24h
          all covariates lagged by the full horizon   pass                                   36 lagged covar

## Question 6

> **Based on accuracy, interpretability, uncertainty, computational cost, and
> ease of deployment, which model would you recommend for practical
> smart-home energy forecasting, and why?**

Five criteria, so the evidence is assembled as a scorecard rather than a
single number.


In [17]:
print("Accuracy")
print("=" * 78)
if master is not None:
    print(master.nsmallest(6, "MAE")[
        ["rank", "model", "MAE", "MASE", "skill vs benchmark %"]]
        .round(2).to_string(index=False))

print("\nStability")
if by_origin is not None:
    stability = (by_origin.groupby("model")["MAE"]
                 .agg(mean="mean", std="std", max="max")
                 .assign(cv=lambda d: d["std"] / d["mean"])
                 .nsmallest(6, "mean"))
    print(stability.round(2).to_string())

print("\nUncertainty quantification")
if interval_comparison is not None:
    print(interval_comparison.round(3).to_string(index=False))
else:
    print("  (run Part 7 to produce the matched-level interval comparison)")

print("\nComputational cost and deployment")
if cost is not None:
    print(cost.to_string(index=False))

print("\nThe common limitation: peak performance")
if peak_performance is not None:
    print(peak_performance.nsmallest(6, "mean_abs_error").round(1).to_string())

print("\nUnexploited structure in the errors")
if error_autocorr is not None:
    print(error_autocorr.to_string(index=False))


Accuracy
 rank                          model   MAE  MASE  skill vs benchmark %
    1               chronos_zeroshot 33.58  0.63                 21.77
    2 hist_gradient_boosting_regimeB 36.55  0.69                 14.84
    3                  random_forest 39.37  0.74                  8.29
    4         hist_gradient_boosting 40.03  0.75                  6.74
    5          seasonal_naive_weekly 42.92  0.80                  0.00
    6           seasonal_naive_daily 48.39  0.91                -12.75

Stability
                                 mean    std    max    cv
model                                                    
chronos_zeroshot                33.58  22.46  83.63  0.67
hist_gradient_boosting_regimeB  36.55  18.79  79.75  0.51
random_forest                   39.37  19.52  80.03  0.50
hist_gradient_boosting          40.03  18.89  78.88  0.47
seasonal_naive_weekly           42.92  28.24  97.64  0.66
seasonal_naive_daily            48.39  24.51  94.38  0.51

Uncertainty quanti

## Cross-cutting evidence

Two findings recur across several questions and are worth stating once,
clearly, since they explain why every model converges to a similar accuracy.


In [18]:
print("1. All models compress towards the mean")
print("=" * 74)
forecast_files = list(FORECAST_DIR.glob("part*_forecasts.csv"))
if forecast_files:
    frames = [pd.read_csv(f, parse_dates=["timestamp"]) for f in forecast_files]
    pooled = pd.concat(frames, ignore_index=True)

    slopes = []
    for model, group in pooled.groupby("model"):
        if len(group) == TEST_STEPS:
            slope = np.polyfit(group["actual"], group["prediction"], 1)[0]
            slopes.append({"model": model, "calibration slope": round(slope, 3)})
    slope_frame = pd.DataFrame(slopes).sort_values("calibration slope",
                                                   ascending=False)
    print(slope_frame.to_string(index=False))
    print("\n  A slope of 1.0 would mean peaks are predicted at full height.")

print("\n2. Variance the seasonal structure cannot explain")
if decomposition is not None:
    print(decomposition.round(3).to_string())
    print("\n  Neither seasonal component exceeds 0.5, so more than half the")
    print("  variance is irregular switching behaviour no seasonal model reaches.")

print(f"\n  Test period: {test_series.mean():.1f} Wh mean, "
      f"{test_series.std():.1f} Wh std, "
      f"{(test_series > 200).sum()} hours above 200 Wh "
      f"({(test_series > 200).mean() * 100:.1f}%)")


1. All models compress towards the mean
                         model  calibration slope
          seasonal_naive_daily              0.374
hist_gradient_boosting_regimeB              0.346
        hist_gradient_boosting              0.310
                 random_forest              0.304
               sarimax_weather              0.291
              sarimax_calendar              0.283
                        sarima              0.281
              chronos_zeroshot              0.228
         seasonal_naive_weekly              0.227
                          mean             -0.000
                         drift             -0.050
                         naive             -0.051

  A slope of 1.0 would mean peaks are predicted at full height.

2. Variance the seasonal structure cannot explain
               strength
seasonal_24h      0.474
seasonal_168h     0.298
trend             0.046

  Neither seasonal component exceeds 0.5, so more than half the
  variance is irregular switching

## Writing the answers

The brief requires these answers in the report, and states plainly that
generative AI must not be used to write it. The numbers above are the
evidence; the argument connecting them has to be written by hand, and that is
where the discussion marks sit.

Three things worth doing in the write-up regardless of what the numbers say:

**Quote uncertainty alongside every comparison.** The gap between the leading
models is small relative to the day-to-day spread. A report that presents a
ranking without acknowledging this is claiming more than the data supports.

**Explain mechanisms, not just magnitudes.** Saying the weekly seasonal naive
beat the daily one is a result; explaining that it does so because it captures
the weekday/weekend split visible in the Part 1 profile is analysis.

**Treat the negative results as findings.** Exogenous variables making the
model worse, and every model under-predicting peaks by a factor of two, are
substantive conclusions about the problem rather than failures of method.
